[Guide Index](README.md)


# 02.07 OWL 2 DL Reasoning

The [Graph Inferencing guide](02.06-graphs-inferencing.ipynb) covers the RDF, RDFS and OWL 2 RL entailment regimes of `infer()`. This guide covers a fourth, `entailment=ENTAILMENT['OWL-Direct']`: OWL 2 DL reasoning, which can use axioms that the rules of OWL 2 RL cannot: unions ("A or B"), complements ("not A") and classes that can have no members. Each is explained in the examples below.

OWL 2 DL needs a separate reasoner to perform its inference, chosen with `engine=`:

- `engine="hermit"` (the default): [HermiT](http://www.hermit-reasoner.com/), through the `owlready2` library. It needs a Java runtime.
- `engine="rustdl"`: [RustDL](https://github.com/MaastrichtU-IDS/rustdl), written in Rust.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom; later cells reuse variables from earlier ones.

The reasoners are in the separate `starlayer-reasoning` package, and HermiT and RustDL are its optional extras. Run the cell below. If they are not installed yet, it installs them (from the checkout when the notebook is inside the repository, otherwise from GitHub) and then stops with a message. When that happens, restart the kernel (Kernel > Restart) and run the notebook again from the top.

HermiT also needs a Java runtime on `PATH`. RustDL needs no Java. If a requested reasoner is missing, `infer()` raises an error that says what to install.

In [1]:
# Install the reasoners if they cannot be imported: starlayer-reasoning with the HermiT and RustDL extras.
# Inside a checkout of the repository it installs from the checkout, otherwise from GitHub.
try:
    import owlready2, rustdl, starlayer_reasoning  # noqa: F401, E401
    print("starlayer-reasoning, owlready2 and rustdl are installed - skipping install.")
except ImportError as exc:
    import pathlib

    print(f"The reasoners need installing ({exc}).")
    checkout = pathlib.Path("../..").resolve()
    if (checkout / "packages" / "starlayer-reasoning").is_dir():
        %pip install -e {checkout}/packages/starlayer -e "{checkout}/packages/starlayer-reasoning[hermit,rustdl]"
    else:
        %pip install --upgrade "starlayer @ git+https://github.com/hidden-graph/starlayer.git@shacl-engine#subdirectory=packages/starlayer" "starlayer-reasoning[hermit,rustdl] @ git+https://github.com/hidden-graph/starlayer.git@shacl-engine#subdirectory=packages/starlayer-reasoning"
    raise RuntimeError("starlayer-reasoning was installed: restart the kernel and run the notebook again.") from None

starlayer-reasoning, owlready2 and rustdl are installed - skipping install.


In [2]:
from starlayer.graph import ENTAILMENT, OWL, RDF, Namespace, StarLayerGraph

EX = Namespace("http://example.org/")

## 02.07.1 A disjunction

`g.infer(entailment=ENTAILMENT['OWL-Direct'])` runs OWL 2 DL reasoning.

In the example, every car is electric or gas-powered (a *union*: a car must belong to at least one of the two classes), and `ex:mycar` is a car that is not gas-powered (a *complement*: it belongs to the class of everything that is not `ex:Gas`). So it must be electric. The reasoner reaches that by trying each branch of the union and finding that the `ex:Gas` branch contradicts the facts. The rules of OWL 2 RL cannot reason by cases like this.

In [ ]:
# Every car is electric or gas-powered: Car is a subclass of the union of Electric and Gas.
# ex:Car rdfs:subClassOf [ owl:unionOf ( ex:Electric ex:Gas ) ] .
# ex:mycar is a car that is not gas-powered: written as it is the complement of Gas.
#   ex:mycar a ex:Car, [ owl:complementOf ex:Gas ] .
DATA = """
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:Car rdfs:subClassOf [ owl:unionOf ( ex:Electric ex:Gas ) ] .
    ex:mycar a ex:Car, [ owl:complementOf ex:Gas ] .
"""

g = StarLayerGraph()
g.parse(data=DATA, format="turtle")

# OWL 2 DL can use those facts to infer that mycar must be electric.
closed = g.infer(entailment=ENTAILMENT['OWL-Direct'])
print("OWL-Direct: mycar a Electric:", (EX.mycar, RDF.type, EX.Electric) in closed)

closed_rl = g.infer(entailment=ENTAILMENT['OWL-RDF-Based'])
print("OWL-RL:     mycar a Electric:", (EX.mycar, RDF.type, EX.Electric) in closed_rl)
print("\nentailed graph")
print(closed.serialize(pretty=True))

OWL-Direct: mycar a Electric: True
OWL-RL:     mycar a Electric: False

entailed graph
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

ex:Car rdfs:subClassOf [
        owl:unionOf ( ex:Electric ex:Gas )
    ] .

ex:mycar a ex:Car, ex:Electric, [
        owl:complementOf ex:Gas
    ] .



## 02.07.2 An inconsistency

Add one more fact: `ex:mycar` is also not electric (`ex:mycar a [ owl:complementOf ex:Electric ]`). But every car must be electric or gas-powered, so the data contradicts itself: the graph is *inconsistent*, meaning there is no possible situation in which all of it is true. OWL 2 DL reasoning raises `InconsistentOntologyError`. OWL 2 RL does not notice.

In [4]:
from starlayer_reasoning.owl_dl import InconsistentOntologyError

bad = StarLayerGraph()
bad.parse(data=DATA + "ex:mycar a [ owl:complementOf ex:Electric ] .", format="turtle")

# OWL 2 RL (the `owlrl` library) reports a contradiction it detects by adding a triple with this predicate.
ERR = Namespace("http://www.daml.org/2002/03/agents/agent-ont#")

# OWL 2 RL does not detect this contradiction.
closed_rl = bad.infer(entailment=ENTAILMENT['OWL-RDF-Based'])
print("OWL-RL flags an inconsistency:", bool(list(closed_rl.objects(None, ERR.error))))

# OWL 2 DL does.
try:
    bad.infer(entailment=ENTAILMENT['OWL-Direct'])
except InconsistentOntologyError as e:
    print("OWL-Direct raises InconsistentOntologyError:\n", e)

OWL-RL flags an inconsistency: False


OWL-Direct raises InconsistentOntologyError:
 infer(entailment=ENTAILMENT['OWL-Direct']) found the graph's data logically inconsistent under OWL 2 DL semantics.


## 02.07.3 An unsatisfiable class

OWL 2 DL can also reason about a class itself. A class is *unsatisfiable* when it can have no members at all, and the reasoner reports it as equivalent to `owl:Nothing`, the class with no members.

Every member of a pricing tier may only have excellent credit profiles (`owl:allValuesFrom`: every value of the property must belong to the class). A second-chance tier, a subclass of a pricing tier, requires at least one prior-default profile (`owl:someValuesFrom`: at least one value must belong to the class). The two credit classes are disjoint (`owl:disjointWith`: nothing belongs to both), so the tier can have no members.

In [5]:
tiers = StarLayerGraph()
tiers.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    ex:hasCreditProfile a owl:ObjectProperty .
    ex:ExcellentCredit owl:disjointWith ex:PriorDefault .
    ex:PricingTier rdfs:subClassOf
        [ a owl:Restriction ; owl:onProperty ex:hasCreditProfile ; owl:allValuesFrom ex:ExcellentCredit ] .
    ex:SecondChanceTier rdfs:subClassOf ex:PricingTier ,
        [ a owl:Restriction ; owl:onProperty ex:hasCreditProfile ; owl:someValuesFrom ex:PriorDefault ] .
""", format="turtle")

# The statement "SecondChanceTier is equivalent to owl:Nothing", written as a (subject, predicate, object) tuple.
unsatisfiable = (EX.SecondChanceTier, OWL.equivalentClass, OWL.Nothing)

# OWL 2 DL infers that statement; OWL 2 RL does not.
closed = tiers.infer(entailment=ENTAILMENT['OWL-Direct'])
closed_rl = tiers.infer(entailment=ENTAILMENT['OWL-RDF-Based'])
print("OWL-Direct: SecondChanceTier is owl:Nothing:", unsatisfiable in closed)
print("OWL-RL:     SecondChanceTier is owl:Nothing:", unsatisfiable in closed_rl)

# Every unsatisfiable class the reasoner found.
print("Unsatisfiable classes:", sorted(c.value.rsplit("/", 1)[-1] for c in closed.subjects(OWL.equivalentClass, OWL.Nothing)))

OWL-Direct: SecondChanceTier is owl:Nothing: True
OWL-RL:     SecondChanceTier is owl:Nothing: False
Unsatisfiable classes: ['SecondChanceTier']


## 02.07.4 Two engines

`engine=` chooses the reasoner. The example in section 02.07.1 gives the same answer with either:

| | `engine="hermit"` (default) | `engine="rustdl"` |
|---|---|---|
| Needs | a Java runtime | nothing beyond `pip install "starlayer-reasoning[rustdl]"` |
| Complete for OWL 2 DL (finds every entailed conclusion) | yes | not guaranteed: RustDL never reports a conclusion that does not follow, but its documentation says it is not provably complete on all of OWL 2 DL |


In [6]:
closed = g.infer(entailment=ENTAILMENT['OWL-Direct'], engine="rustdl")
print("rustdl: mycar a Electric:", (EX.mycar, RDF.type, EX.Electric) in closed)

rustdl: mycar a Electric: True


## 02.07.5 Materialize once, query many times

OWL 2 DL reasoning costs far more time than the rules of OWL 2 RL (see [Appendix B2](appendix-b2-dl-reasoning-examples.ipynb) for how steeply time grows with the size of the graph). For many queries on data that does not change, run `infer()` once and query the closure (the original graph plus everything the reasoner derived) with ordinary SPARQL.

In [7]:
closure = g.infer(entailment=ENTAILMENT['OWL-Direct'])
rows = closure.query("PREFIX ex: <http://example.org/> SELECT ?x WHERE { ex:mycar a ?x FILTER(isIRI(?x)) }")
print("mycar's entailed types:", sorted(row["x"].value.rsplit("/", 1)[-1] for row in rows))

mycar's entailed types: ['Car', 'Electric']


## Appendix

[Appendix B2](appendix-b2-dl-reasoning-examples.ipynb) includes more complex OWL DL examples.